In [ ]:
# =========================
# RQ2: Model Comparison
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb

# -------------------------
# 1. Load Data
# -------------------------
df = pd.read_csv(
    "/kaggle/input/datasets/sharmajicoder/gaming-and-mental-health/gaming_mental_health_10M_40features.csv"
)

df = df.dropna()
df = df.sample(n=20000, random_state=42)

TARGET = df.columns[-1]

# -------------------------
# 2. Prepare Features and Target
# -------------------------
X = pd.get_dummies(df.drop(TARGET, axis=1), drop_first=True).astype(float)

le = LabelEncoder()
y = le.fit_transform(df[TARGET])
y = (y > 0).astype(int)

# -------------------------
# 3. Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# 4. Models
# -------------------------
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="mlogloss",
        n_jobs=-1
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

# -------------------------
# 5. Train and Evaluate
# -------------------------
results = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    # AUC for binary or multi-class
    try:
        y_proba = model.predict_proba(X_test)

        if len(np.unique(y)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(
                y_test,
                y_proba,
                multi_class="ovr",
                average="weighted"
            )
    except:
        auc = np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "AUC": auc
    })

# -------------------------
# 6. Save Table
# -------------------------
table = pd.DataFrame(results)
table.to_csv("RQ2_table.csv", index=False)

print("\n=== RQ2 Model Comparison Table ===")
print(table)

best_model = table.sort_values("F1-score", ascending=False).iloc[0]
print("\nBest model based on F1-score:")
print(best_model["Model"])

# -------------------------
# 7. Plot Grouped Bar Chart
# -------------------------
metrics = ["Accuracy", "Precision", "Recall", "F1-score", "AUC"]

x = np.arange(len(table["Model"]))
width = 0.15

plt.figure(figsize=(10, 6))

for i, metric in enumerate(metrics):
    values = table[metric].fillna(0)

    bars = plt.bar(
        x + (i - 2) * width,
        values,
        width,
        label=metric
    )

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.01,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.title("RQ2: Model Comparison", fontsize=14, fontweight="bold")
plt.xlabel("Model", fontsize=12, fontweight="bold")
plt.ylabel("Score", fontsize=12, fontweight="bold")

plt.xticks(x, table["Model"])
plt.ylim(0, 1.05)

plt.legend(loc="upper center", ncol=5)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("RQ2_figure.pdf", bbox_inches="tight")
plt.show()